# Guide des métamodèles `scikit-learn` : les commandes essentielles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dbetteb/early-ML/blob/master/08_CAS_TEST_SER/Guide_metamodeles_sklearn_SER.ipynb)

Ce notebook est un **aide-mémoire** : pour chaque famille de métamodèles, on rappelle le principe, les commandes, les hyperparamètres qui
comptent et les pièges classiques. Après chaque fiche, une cellule vide vous attend : construisez le modèle et testez-le sur le cas SER
(111 simulations, 11 entrées dans $[0,1]$, une sortie $y$).

Les outils communs (données, validation croisée, fonction d'évaluation) sont fournis et prêts à l'emploi : vous n'écrivez que les modèles.
À la fin, un tableau compare tous les modèles que vous aurez construits.

La correction complète est le notebook `Benchmark_metamodeles_SER.ipynb`.

**Sommaire**
0. Outils communs : données, découpage, validation croisée, réglage des hyperparamètres
1. Modèles linéaires : MCO, Ridge, Lasso, Huber, PLS, polynômes
2. Méthodes à noyau : SVR, Kernel Ridge, processus gaussien
3. $k$ plus proches voisins
4. Réseau de neurones (MLP)
5. Ensembles d'arbres : Random Forest, Extra Trees, Gradient Boosting
6. Comparaison finale

## 0. Outils communs

### 0.1 Données

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split, KFold, RepeatedKFold, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import TransformedTargetRegressor
from sklearn.metrics import r2_score, mean_squared_error
warnings.filterwarnings("ignore")

FICHIER = "data_scaled.csv"
URL = "https://raw.githubusercontent.com/dbetteb/early-ML/master/08_CAS_TEST_SER/" + FICHIER
data = pd.read_csv(FICHIER if os.path.exists(FICHIER) else URL, sep=";")
y = data.pop("y").to_numpy()
X = data.to_numpy()
n, d = X.shape
print(f"{n} simulations, {d} entrées : {list(data.columns)}")

### 0.2 Les commandes de base, communes à tous les modèles

Tous les estimateurs `scikit-learn` s'utilisent de la même façon :

```python
modele = UneClasse(param1=..., param2=...)   # 1. créer (rien n'est appris)
modele.fit(X_app, y_app)                      # 2. apprendre
y_pred = modele.predict(X_test)               # 3. prédire
modele.get_params()                           # tous les hyperparamètres
modele.set_params(param1=...)                 # en changer un
clone(modele)                                 # copie neuve, non entraînée
```

**Découpage apprentissage / test**, pour une première évaluation (une seule coupe, donc un résultat bruité avec 111 points) :

```python
X_app, X_test, y_app, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
r2_score(y_test, y_pred)                          # R² sur le test
np.sqrt(mean_squared_error(y_test, y_pred))       # RMSE
```

**Validation croisée**:

```python
cv = KFold(n_splits=5, shuffle=True, random_state=0)
cross_val_score(modele, X, y, cv=cv, scoring="r2")       # un score par pli
y_oof = cross_val_predict(modele, X, y, cv=cv)           # prédictions hors échantillon
q2 = r2_score(y, y_oof)                                  # Q² : coefficient de prédictivité
```

**Réglage des hyperparamètres par grille** : `GridSearchCV` teste toutes les combinaisons par validation croisée, puis
ré-apprend le meilleur modèle sur toutes les données qu'on lui donne. C'est lui-même un estimateur (`fit`, `predict`) :

```python
grille = {"C": [0.1, 1, 10, 100], "gamma": [0.01, 0.1, 1]}
recherche = GridSearchCV(SVR(), grille, cv=5, scoring="neg_mean_squared_error")
recherche.fit(X_app, y_app)
recherche.best_params_, recherche.best_score_            # meilleur réglage et son score de CV
pd.DataFrame(recherche.cv_results_)                       # tous les résultats
```

> Ne jamais régler les hyperparamètres sur les données de test : le score de test serait optimiste.
> Dans la fonction `evaluer` ci-dessous, `GridSearchCV` est relancé **dans chaque pli** (validation croisée imbriquée).

**Enchaîner des transformations** : `make_pipeline` colle des étapes de prétraitement et un modèle en un seul estimateur.
Les paramètres d'une étape se désignent par `nomdeletape__parametre` (nom de la classe en minuscules) :

```python
pipe = make_pipeline(PolynomialFeatures(degree=2), StandardScaler(), Ridge())
GridSearchCV(pipe, {"ridge__alpha": [0.01, 0.1, 1]}, cv=5)
```

**Standardiser la sortie** : certains hyperparamètres dépendent de l'échelle de $y$ (`epsilon` du SVR, pénalisation d'un MLP).
`TransformedTargetRegressor` standardise $y$ avant `fit` et revient à l'échelle d'origine dans `predict`.
Les paramètres du modèle enveloppé prennent alors le préfixe `regressor__` :

```python
y_standardise(SVR())                                     # fonction fournie ci-dessous
GridSearchCV(y_standardise(SVR()), {"regressor__C": [1, 10, 100]}, cv=5)
```

### 0.3 Fonctions fournies

* `y_standardise(modele)` : enveloppe un modèle pour standardiser la sortie ;
* `cv_interne` : découpage en 5 plis à utiliser dans vos `GridSearchCV` (`cv=cv_interne`) ;
* `evaluer(modele, R=3)` : validation croisée 5 plis répétée `R` fois ; affiche le $Q^2$ moyen (± écart-type entre répétitions),
  la RMSE et le temps de calcul. Le modèle est ré-entraîné (et ré-réglé) dans chaque pli ;
* `modeles` : dictionnaire où ranger vos modèles pour la comparaison finale.

In [ ]:
def y_standardise(modele):
    """Standardise la sortie avant l'apprentissage, revient à l'échelle d'origine en prédiction."""
    return TransformedTargetRegressor(regressor=modele, transformer=StandardScaler())

cv_interne = KFold(5, shuffle=True, random_state=0)

def evaluer(modele, R=3, afficher=True):
    """Validation croisée 5 plis répétée R fois. Renvoie les Q2 et RMSE de chaque répétition."""
    t0 = time.perf_counter()
    q2, rmse = [], []
    for r in range(R):
        y_oof = cross_val_predict(clone(modele), X, y, cv=KFold(5, shuffle=True, random_state=100 + r))
        q2.append(r2_score(y, np.ravel(y_oof)))
        rmse.append(np.sqrt(mean_squared_error(y, np.ravel(y_oof))))
    if afficher:
        print(f"Q2 = {np.mean(q2):.3f} ± {np.std(q2):.3f}   RMSE = {np.mean(rmse):.3f}   "
              f"({time.perf_counter() - t0:.1f} s pour {5 * R} apprentissages)")
    return np.array(q2), np.array(rmse)

modeles = {}

Exemple complet avec le modèle le plus simple :

In [ ]:
from sklearn.linear_model import LinearRegression

modeles["Linéaire (MCO)"] = LinearRegression()
evaluer(modeles["Linéaire (MCO)"]);

# coefficients appris sur toutes les données
pd.Series(LinearRegression().fit(X, y).coef_, index=data.columns).round(3)

---
## 1. Modèles linéaires

$$\hat f(\mathbf x) = \beta_0 + \sum_{j=1}^d \beta_j x_j,$$
éventuellement sur des variables transformées (puissances, produits). Les variantes diffèrent par la **perte** et la **pénalisation** des coefficients.

### 1.1 Ridge et Lasso : moindres carrés pénalisés

$$\min_\beta \|y - X\beta\|^2 + \alpha \|\beta\|_2^2 \ \text{(Ridge)}, \qquad \min_\beta \tfrac{1}{2n}\|y - X\beta\|^2 + \alpha \|\beta\|_1 \ \text{(Lasso)}.$$

```python
from sklearn.linear_model import Ridge, RidgeCV, Lasso, LassoCV

Ridge(alpha=1.0)                                    # alpha fixé
RidgeCV(alphas=np.logspace(-4, 3, 30))              # alpha choisi par validation croisée interne (efficace)
LassoCV(alphas=np.logspace(-4, 1, 50), cv=cv_interne, max_iter=50_000)  # 50 valeurs d'alpha testées automatiquement

m = RidgeCV(alphas=np.logspace(-4, 3, 30)).fit(X, y)
m.alpha_, m.coef_                                   # alpha retenu, coefficients
```

| paramètre | rôle | valeurs typiques |
|---|---|---|
| `alpha` / `alphas` | force de la pénalisation : 0 = moindres carrés | grille logarithmique, de $10^{-4}$ à $10^{3}$ |
| `max_iter` (Lasso) | itérations de la descente par coordonnées | augmenter si avertissement de non-convergence |

**Pièges.** La pénalisation agit sur des coefficients dont l'échelle dépend de celle des entrées : standardiser les entrées si elles ne sont
pas déjà comparables (ici elles sont toutes dans $[0,1]$). Le Lasso met des coefficients **exactement à 0** : c'est une sélection de variables.

**À vous.** Construisez `modeles["Ridge"]`, puis évaluez-le avec `evaluer(modeles["Ridge"])`.

In [ ]:
# votre code ici

**À vous.** Construisez `modeles["Lasso"]`, puis évaluez-le avec `evaluer(modeles["Lasso"])`.
regardez quels coefficients sont nuls.

In [ ]:
# votre code ici

### 1.3 PLS : régression sur composantes latentes

Projette les entrées sur quelques directions qui expliquent au mieux la sortie, puis régresse sur ces composantes.

```python
from sklearn.cross_decomposition import PLSRegression

GridSearchCV(PLSRegression(scale=False), {"n_components": [1, 2, 3, 4, 6, 8, 11]}, cv=cv_interne)
```

| paramètre | rôle | valeurs typiques |
|---|---|---|
| `n_components` | nombre de composantes latentes (`d` = moindres carrés) | de 1 à $d$, par CV |
| `scale` | standardise les entrées | `False` ici (déjà dans $[0,1]$) |

**À vous.** Construisez `modeles["PLS"]`, puis évaluez-le avec `evaluer(modeles["PLS"])`.


In [ ]:
# votre code ici

### 1.4 Polynômes : linéaire sur des variables transformées

`PolynomialFeatures` fabrique les carrés et les produits de variables ; on régresse ensuite linéairement.

```python
from sklearn.preprocessing import PolynomialFeatures

PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)   # x_i et x_i x_j : d + d(d-1)/2 = 66 termes
PolynomialFeatures(degree=2, include_bias=False)                          # + les carrés x_i^2 : 77 termes

make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler(), RidgeCV(alphas=np.logspace(-4, 3, 30)))
```

**Pièges.** Avec 89 points d'apprentissage risque d'overfitting : excellent $R^2$ d'apprentissage,
mauvais $Q^2$. La pénalisation est alors indispensable

**À vous.** Comparez le polynôme avec interactions **sans pénalisation** (le métamodèle des notebooks d'AS) et le polynôme complet
avec `RidgeCV`. Rangez-les dans `modeles["Poly2 interactions (MCO)"]` et `modeles["Poly2 + Ridge"]`.
Comparez aussi le $R^2$ d'apprentissage (`modele.fit(X, y).score(X, y)`) au $Q^2$.

In [ ]:
# votre code ici

---
## 2. Méthodes à noyau

Un noyau $k(\mathbf x, \mathbf x')$ mesure la ressemblance entre deux points ; la prédiction est une combinaison de noyaux centrés sur les
points d'apprentissage : $\hat f(\mathbf x) = \sum_i a_i\, k(\mathbf x, \mathbf x_i) + b$.
Noyau gaussien (RBF) : $k(\mathbf x, \mathbf x') = \exp(-\gamma \|\mathbf x - \mathbf x'\|^2)$.

### 2.1 SVR : machine à vecteurs supports pour la régression

Perte **$\varepsilon$-insensible** : les erreurs inférieures à $\varepsilon$ ne coûtent rien, au-delà la perte est linéaire.
Seuls les points hors du tube (les **vecteurs supports**) déterminent le modèle.

```python
from sklearn.svm import SVR

SVR(kernel="rbf", C=10, gamma=0.1, epsilon=0.1)

grille = {"regressor__C": np.logspace(-1, 3, 5),
          "regressor__gamma": np.logspace(-3, 0, 4),
          "regressor__epsilon": [0.01, 0.1]}
GridSearchCV(y_standardise(SVR(kernel="rbf")), grille, cv=cv_interne)

m.best_params_                                # (après fit) réglage retenu
m.best_estimator_.regressor_.support_         # indices des vecteurs supports
```

| paramètre | rôle | valeurs typiques |
|---|---|---|
| `kernel` | forme du noyau | `"rbf"` (défaut), `"linear"`, `"poly"` |
| `C` | compromis régularité / attache aux données : grand `C` = suit les données | log, de 0,1 à 1000 |
| `gamma` | inverse du carré de la portée du noyau : grand `gamma` = noyau très local | log, de $10^{-3}$ à 1 ; défaut `"scale"` = $1/(d\,\mathrm{Var}(X))$ |
| `epsilon` | demi-largeur du tube insensible, **dans les unités de $y$** | 0,01 à 0,1 sur $y$ standardisé |

**Pièges.** Sans standardiser $y$, `epsilon` n'a pas de sens universel. Les trois paramètres interagissent : toujours les régler ensemble,
par grille. Si le meilleur réglage tombe au bord de la grille, élargir la grille.

**À vous.** Construisez `modeles["SVR (RBF)"]`, puis évaluez-le avec `evaluer(modeles["SVR (RBF)"])`.


In [ ]:
# votre code ici

### 2.2 Kernel Ridge : ridge dans l'espace du noyau

Même forme de prédiction que le SVR, mais avec la perte **quadratique** : solution explicite, pas de vecteurs supports.
Sa prédiction coïncide avec la moyenne d'un processus gaussien de même noyau.

```python
from sklearn.kernel_ridge import KernelRidge

GridSearchCV(y_standardise(KernelRidge(kernel="rbf")),
             {"regressor__alpha": np.logspace(-4, 0, 5), "regressor__gamma": np.logspace(-3, 0, 4)}, cv=cv_interne)
```

| paramètre | rôle | valeurs typiques |
|---|---|---|
| `alpha` | pénalisation (joue le rôle de $1/C$) | log, de $10^{-4}$ à 1 |
| `gamma` | comme pour le SVR | log, de $10^{-3}$ à 1 |

**À vous.** Construisez `modeles["Kernel Ridge (RBF)"]`, puis évaluez-le avec `evaluer(modeles["Kernel Ridge (RBF)"])`.
comparez au SVR.

In [ ]:
# votre code ici

### 2.3 Processus gaussien (krigeage)

Modèle probabiliste : la prédiction est une **loi gaussienne** (moyenne et écart-type). Les hyperparamètres du noyau sont réglés par
**maximum de vraisemblance**, sans `GridSearchCV`.

```python
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF, WhiteKernel

noyau = (ConstantKernel(1.0)                                                  # variance du processus
         * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e3), nu=2.5)   # une portée par entrée (ARD)
         + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-8, 1.0)))     # pépite (bruit, régularisation)
gp = GaussianProcessRegressor(noyau, normalize_y=True, n_restarts_optimizer=3, random_state=0)

gp.fit(X, y)
gp.kernel_                                     # noyau avec les hyperparamètres estimés
gp.kernel_.k1.k2.length_scale                  # portées estimées : petite portée = entrée influente
moyenne, ecart_type = gp.predict(X_nouveaux, return_std=True)
gp.log_marginal_likelihood_value_
```

| paramètre | rôle | valeurs typiques |
|---|---|---|
| noyau | régularité supposée de la fonction | `Matern(nu=2.5)` (deux fois dérivable), `RBF` (infiniment lisse) |
| `length_scale=np.ones(d)` | une portée par entrée (*ARD*) au lieu d'une portée commune | toujours en métamodélisation |
| `WhiteKernel` | bruit ou erreur de modèle ; stabilise l'inversion | indispensable si la sortie est bruitée |
| `normalize_y` | centre et réduit $y$ | `True` |
| `n_restarts_optimizer` | redémarrages de l'optimisation de la vraisemblance (optima locaux) | 3 à 10 |

**Pièges.** Des portées collées aux bornes (`length_scale_bounds`) signalent une entrée sans influence (portée maximale) ou un problème
de réglage. Vérifier `gp.kernel_`

**À vous.** Construisez `modeles["Processus gaussien"]`, puis évaluez-le avec `evaluer(modeles["Processus gaussien"])`.
Ordre de grandeur atteignable : $Q^2 \approx 0{,}92$ ; affichez les portées estimées et comparez-les aux indices de Sobol' du notebook d'AS.

In [ ]:
# votre code ici

---
## 3. $k$ plus proches voisins

La prédiction est la moyenne des sorties des $k$ points d'apprentissage les plus proches. Aucun apprentissage à proprement parler.

```python
from sklearn.neighbors import KNeighborsRegressor

GridSearchCV(KNeighborsRegressor(), {"n_neighbors": [2, 3, 5, 8, 12], "weights": ["uniform", "distance"]}, cv=cv_interne)
```

| paramètre | rôle | valeurs typiques |
|---|---|---|
| `n_neighbors` | nombre de voisins : petit = prédiction irrégulière | 2 à 15 |
| `weights` | `"distance"` donne plus de poids aux voisins proches | `"uniform"` ou `"distance"` |
| `p` | distance de Minkowski (2 = euclidienne) | 1 ou 2 |

**Pièges.** Fléau de la dimension : en dimension 11 avec 89 points, même le plus proche voisin est loin. Toutes les entrées comptent
autant dans la distance, influentes ou non.

**À vous.** Construisez `modeles["k plus proches voisins"]`, puis évaluez-le avec `evaluer(modeles["k plus proches voisins"])`.


In [ ]:
# votre code ici

---
## 4. Réseau de neurones : perceptron multicouche (MLP)

$\hat f(\mathbf x) = \mathbf w_2^\top \sigma(W_1 \mathbf x + \mathbf b_1) + b_2$ pour une couche cachée, avec une fonction
d'activation $\sigma$ (`tanh`, `relu`). Les poids sont appris en minimisant l'erreur quadratique pénalisée.

```python
from sklearn.neural_network import MLPRegressor

y_standardise(MLPRegressor(hidden_layer_sizes=(10,), activation="tanh", solver="lbfgs",
                           alpha=0.05, max_iter=10_000, random_state=0))

# réglage par grille
grille = {"regressor__hidden_layer_sizes": [(5,), (10,), (20,)], "regressor__alpha": [1e-3, 1e-2, 0.05, 0.1]}
GridSearchCV(y_standardise(MLPRegressor(activation="tanh", solver="lbfgs", max_iter=10_000, random_state=0)),
             grille, cv=cv_interne)

m.coefs_, m.intercepts_          # (sur le MLP entraîné) poids de chaque couche
m.n_iter_                        # nombre d'itérations effectuées
# m.loss_curve_                  # courbe d'apprentissage : n'existe qu'avec solver="adam" ou "sgd"
```

| paramètre | rôle | valeurs typiques |
|---|---|---|
| `hidden_layer_sizes` | neurones par couche cachée : `(10,)` = une couche de 10 ; `(20, 10)` = deux couches | `(5,)` à `(20,)` pour 100 points |
| `activation` | non-linéarité | `"tanh"` (fonctions régulières), `"relu"` |
| `solver` | algorithme d'optimisation | `"lbfgs"` pour les petits jeux de données ; `"adam"` pour les gros |
| `alpha` | pénalisation $L_2$ des poids (*weight decay*) | $10^{-3}$ à 0,1 |
| `max_iter` | itérations maximales | grand avec `lbfgs` (10 000) |
| `random_state` | initialisation des poids | à fixer : le problème n'est pas convexe |
| `early_stopping` | arrêt quand le score de validation ne progresse plus (`adam` / `sgd`) | utile pour les gros jeux de données |

**Pièges.** Une couche de 10 neurones a $(d+1)\times 10 + 11 = 131$ poids pour 89 points : sans `alpha`, sur-apprentissage garanti.
Ne pas oublier de standardiser $y$. Deux initialisations différentes donnent deux modèles différents : tester plusieurs `random_state`.

**À vous.** Construisez `modeles["MLP"]`, puis évaluez-le avec `evaluer(modeles["MLP"])`.
essayez `solver="adam"` et comparez.

In [ ]:
# votre code ici

---
## 5. Ensembles d'arbres

Un arbre de régression découpe l'espace en pavés et prédit une constante par pavé. Seul, il est instable ; on en combine beaucoup.

* **Random Forest** : moyenne de nombreux arbres profonds, chacun appris sur un échantillon bootstrap ;
* **Extra Trees** : idem, avec des seuils de coupure tirés au hasard (prédictions plus lisses) ;
* **Gradient Boosting** : arbres peu profonds ajoutés **séquentiellement**, chacun corrige les résidus des précédents.

```python
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor,
                              GradientBoostingRegressor, HistGradientBoostingRegressor)

RandomForestRegressor(n_estimators=500, max_features=1.0, min_samples_leaf=1, random_state=0, n_jobs=-1)
ExtraTreesRegressor(n_estimators=500, random_state=0, n_jobs=-1)
GradientBoostingRegressor(n_estimators=500, learning_rate=0.03, max_depth=2, subsample=0.8, random_state=0)
HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, min_samples_leaf=5, random_state=0)

m.feature_importances_           # (après fit) importance des variables, selon la réduction d'impureté
```

| modèle | paramètre | rôle | valeurs typiques |
|---|---|---|---|
| RF, ET | `n_estimators` | nombre d'arbres : plus ne nuit jamais, coûte seulement du calcul | 300 à 1000 |
| RF, ET | `max_features` | part des variables tirées à chaque coupure | `1.0` (défaut en régression), `"sqrt"`, 0,5 |
| RF, ET | `min_samples_leaf` | taille minimale d'une feuille | 1 à 5 |
| GB | `learning_rate` × `n_estimators` | pas d'apprentissage et nombre d'arbres se compensent | 0,03 × 500, 0,1 × 150 |
| GB | `max_depth` | profondeur des arbres : 2 = interactions d'ordre 2 | 2 à 4 |
| GB | `subsample` | part des données par arbre (< 1 : boosting stochastique) | 0,8 |
| HGB | `min_samples_leaf` | **défaut 20, trop grand pour 89 points** | 5 |

**Pièges.** Prédiction constante par morceaux : pour une réponse régulière et peu de points, ces méthodes sont souvent battues par les
modèles linéaires et à noyau. `feature_importances_` est biaisée en faveur des variables continues à nombreuses valeurs ; préférer
`sklearn.inspection.permutation_importance` pour interpréter.

**À vous.** Construisez `modeles["Random Forest"]`, puis évaluez-le avec `evaluer(modeles["Random Forest"])`.

In [ ]:
# votre code ici

**À vous.** Construisez `modeles["Gradient Boosting"]`, puis évaluez-le avec `evaluer(modeles["Gradient Boosting"])`.
faites varier `learning_rate` et `n_estimators` ensemble.

In [ ]:
# votre code ici

---
## 6. Comparaison finale

La cellule suivante évalue tous les modèles du dictionnaire `modeles` avec 10 répétitions, puis les classe.
Les valeurs de référence sont celles de la correction (`Benchmark_metamodeles_SER.ipynb`).

In [ ]:
Q2_REFERENCE = {"SVR (RBF)": 0.948, "Kernel Ridge (RBF)": 0.935, "Linéaire (MCO)": 0.934,
                "Ridge": 0.934, "Lasso": 0.934, "PLS": 0.934, "Poly2 + Ridge": 0.924, "Processus gaussien": 0.921,
                "MLP": 0.909, "Gradient Boosting": 0.841, "Poly2 interactions (MCO)": 0.714,
                "Random Forest": 0.708, "k plus proches voisins": 0.566}

bilan = {}
for nom, modele in modeles.items():
    print(f"{nom:30s}", end=" ")
    q2, rmse = evaluer(modele, R=10)
    bilan[nom] = {"Q2": q2.mean(), "écart-type": q2.std(), "RMSE": rmse.mean(), "référence": Q2_REFERENCE.get(nom, np.nan)}

bilan = pd.DataFrame(bilan).T.sort_values("Q2", ascending=False)
bilan["Q2"].plot.barh(xerr=bilan["écart-type"], figsize=(7, 0.4 * len(bilan) + 1), color="steelblue")
plt.gca().invert_yaxis(); plt.xlabel("$Q^2$ (validation croisée 5 plis × 10)"); plt.xlim(left=max(0, bilan["Q2"].min() - 0.05))
plt.tight_layout(); plt.show()
bilan.round(3)

**Pour aller plus loin**

* Tracer les prédictions hors échantillon (`cross_val_predict`) contre les observations : des simulations sont-elles mal prédites par
  tous les modèles ?
* Les écarts entre les meilleurs modèles sont-ils significatifs ?


**Documentation** : [guide utilisateur `scikit-learn`](https://scikit-learn.org/stable/user_guide.html) ;
C. E. Rasmussen, C. K. I. Williams (2006), *Gaussian Processes for Machine Learning*, MIT Press ;
T. Hastie, R. Tibshirani, J. Friedman (2009), *The Elements of Statistical Learning*, 2e éd., Springer.